# 🐘 Lab Semana 1 — Tipos de Datos y Arquitectura Big Data
## Big Data DD283 | Universidad Autónoma del Perú | 2026-1

---
**Nombre**: Raúl Ferreyra  
**Grupo de proyecto**: Grupo 04  
**Empresa/sector del proyecto**: Universidad Norbert Wiener  
**Fecha**: 14/06/2026

---

### 🎯 Objetivos del Lab
Al completar este laboratorio podrás:
1. Cargar y explorar datos **estructurados** con Pandas
2. Procesar datos **semi-estructurados** (JSON)
3. Analizar datos **no estructurados** (texto)
4. Calcular métricas de las **5 V's** del Big Data
5. Diseñar una **arquitectura Big Data** básica para tu proyecto

### ⏱️ Tiempo estimado: 90 minutos

### 📋 Instrucciones
- Ejecuta cada celda en orden (Shift+Enter)
- Donde veas `# 📝 TU CÓDIGO AQUÍ`, escribe tu implementación
- La última celda es una reflexión personal — escríbela con honestidad
- Sube el notebook completo (con outputs) a tu PR de GitHub

In [ ]:
# ── Celda 0: Instalación (solo si estás en Google Colab) ──────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pandas matplotlib seaborn -q
    print('✅ Librerías instaladas en Colab')
else:
    print('✅ Usando entorno local (conda bigdata-ua)')

In [ ]:
# ── Celda 1: Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime, timedelta

# Estilo de gráficos
plt.style.use('dark_background')
sns.set_palette('husl')

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'Python  : {sys.version.split()[0]}')
print('✅ Entorno listo para el lab S1!')

---
## PARTE 1: Datos Estructurados (25 min)
### Dataset: Transacciones de empresa peruana (100K registros)

In [ ]:
# ── Celda 2: Generar dataset estructurado ─────────────────────
np.random.seed(42)
N = 100_000  # 100 mil registros

departamentos = ['Lima', 'Arequipa', 'Cusco', 'Trujillo', 'Piura', 'Chiclayo', 'Iquitos', 'Huancayo']
categorias    = ['Alimentos', 'Electrodomésticos', 'Ropa', 'Farmacia', 'Tecnología', 'Deportes']
metodos_pago  = ['Yape', 'Plin', 'Tarjeta', 'Efectivo', 'BIM']

df = pd.DataFrame({
    'id':          range(1, N+1),
    'fecha':       pd.date_range('2024-01-01', periods=N, freq='5min'),
    'cliente_id':  np.random.randint(1000, 9999, N),
    'depto':       np.random.choice(departamentos, N, p=[.45,.12,.10,.09,.08,.07,.05,.04]),
    'categoria':   np.random.choice(categorias, N),
    'monto':       np.round(np.random.exponential(180, N) + 20, 2),
    'metodo_pago': np.random.choice(metodos_pago, N, p=[.30,.20,.25,.20,.05]),
    'es_fraude':   np.random.choice([0,1], N, p=[.98,.02]),
})

print(f'📊 Dataset creado: {df.shape[0]:,} registros × {df.shape[1]} columnas')
print(f'💾 Tamaño en memoria: {df.memory_usage(deep=True).sum()/1024:.1f} KB')
df.head(3)

In [ ]:
# ── Celda 3: Exploración básica ───────────────────────────────
print('=== INFORMACIÓN GENERAL ===')
df.info()
print()
print('=== ESTADÍSTICAS DESCRIPTIVAS ===')
df[['monto', 'es_fraude']].describe()

In [ ]:
# ── Celda 4: Análisis de las 5 V's ────────────────────────────
print('=== ANÁLISIS DE LAS 5 V\u2019S ===')

# VOLUMEN
mem_kb = df.memory_usage(deep=True).sum() / 1024
print(f'📦 VOLUMEN:')
print(f'   Registros:  {len(df):,}')
print(f'   En memoria: {mem_kb:.1f} KB')
print(f'   Escalando a 100M registros → {mem_kb*1000/1024**2:.1f} GB')

# VELOCIDAD
trans_dia = df.groupby(df['fecha'].dt.date).size()
print(f'\n⚡ VELOCIDAD:')
print(f'   Transacciones/día promedio: {trans_dia.mean():.0f}')
print(f'   Pico máximo en un día:      {trans_dia.max()}')

# VERACIDAD
print(f'\n✅ VERACIDAD:')
print(f'   Nulos por columna:')
print(df.isnull().sum().to_string())
print(f'   Tasa de fraude: {df["es_fraude"].mean()*100:.2f}%')

In [ ]:
# ── Celda 5: Visualización ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Dashboard Transacciones — Empresa Retail Peruana', fontsize=14, fontweight='bold', color='white')

# Gráfico 1: Ventas por departamento
ventas_depto = df.groupby('depto')['monto'].sum().sort_values(ascending=True)
axes[0,0].barh(ventas_depto.index, ventas_depto.values, color='#EB7B2F')
axes[0,0].set_title('Ventas por Departamento', color='white')
axes[0,0].set_xlabel('Monto Total (S/.)', color='white')

# Gráfico 2: Métodos de pago
mp = df['metodo_pago'].value_counts()
colors = ['#EB7B2F','#4BACC6','#70AD47','#FFD700','#C0504D']
axes[0,1].pie(mp.values, labels=mp.index, autopct='%1.1f%%', colors=colors)
axes[0,1].set_title('Métodos de Pago', color='white')

# Gráfico 3: Ventas por categoría
cat_ventas = df.groupby('categoria')['monto'].sum().sort_values()
axes[1,0].barh(cat_ventas.index, cat_ventas.values, color='#4BACC6')
axes[1,0].set_title('Ventas por Categoría', color='white')

# Gráfico 4: Tendencia mensual
monthly = df.groupby(df['fecha'].dt.month)['monto'].sum()
axes[1,1].plot(monthly.index, monthly.values, 'o-', color='#70AD47', linewidth=2, markersize=8)
axes[1,1].set_title('Tendencia Mensual de Ventas', color='white')
axes[1,1].set_xlabel('Mes', color='white')
axes[1,1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('dashboard_s1.png', dpi=150, bbox_inches='tight', facecolor='#0D1B3E')
plt.show()
print('✅ Dashboard guardado como dashboard_s1.png')

---
## PARTE 2: Datos Semi-Estructurados — JSON (15 min)

In [ ]:
# ── Celda 6: Trabajar con JSON ────────────────────────────────
# Simula la respuesta de una API de clientes (como la API de Reniec o SUNAT)

api_response = '''
[
  {"id": "CLI-001", "nombre": "Juan Quispe", "dni": "45678901",
   "contacto": {"email": "j.quispe@empresa.pe", "tel": "987654321", "distrito": "San Miguel"},
   "compras": [150.5, 320.0, 89.9], "activo": true, "tags": ["premium", "digital"]},
  {"id": "CLI-002", "nombre": "Maria Torres", "dni": "56789012",
   "contacto": {"email": null, "tel": "976543210", "distrito": "Miraflores"},
   "compras": [500.0, 1200.5], "activo": true, "tags": ["vip"]},
  {"id": "CLI-003", "nombre": "Carlos Mamani", "dni": "67890123",
   "contacto": {"email": "c.mamani@gmail.com", "tel": null, "distrito": "Callao"},
   "compras": [], "activo": false, "tags": ["nuevo", "inactivo"]}
]
'''

clientes = json.loads(api_response)
print(f'Clientes recibidos: {len(clientes)}')
print(f'\nEstructura del primer cliente:')
print(json.dumps(clientes[0], indent=2, ensure_ascii=False))

In [ ]:
# ── Celda 7: Normalizar JSON a DataFrame ──────────────────────
registros = []
for c in clientes:
    registros.append({
        'id': c['id'],
        'nombre': c['nombre'],
        'distrito': c['contacto']['distrito'],
        'tiene_email': c['contacto']['email'] is not None,
        'tiene_tel': c['contacto']['tel'] is not None,
        'num_compras': len(c['compras']),
        'total_compras': sum(c['compras']),
        'activo': c['activo'],
        'es_premium': any(t in ['premium','vip'] for t in c['tags'])
    })

df_clientes = pd.DataFrame(registros)
print('=== CLIENTES NORMALIZADOS ===')
print(df_clientes.to_string(index=False))
print(f'\n📊 Completitud de datos:')
print(f'  Con email: {df_clientes["tiene_email"].sum()}/{len(df_clientes)}')
print(f'  Con tel:   {df_clientes["tiene_tel"].sum()}/{len(df_clientes)}')
print(f'  Activos:   {df_clientes["activo"].sum()}/{len(df_clientes)}')

---
## PARTE 3: Datos No Estructurados — Análisis de Texto (20 min)

In [ ]:
# ── Celda 8: Análisis básico de sentimiento en comentarios ────
comentarios = [
    {'id': 1, 'texto': 'Excelente servicio en Plaza Vea, muy rápido el pago con Yape!', 'fuente': 'Google'},
    {'id': 2, 'texto': 'Pésima atención, esperé 45 minutos para ser atendido en Tottus', 'fuente': 'Facebook'},
    {'id': 3, 'texto': 'El producto llegó a tiempo pero la caja estaba dañada. Regular', 'fuente': 'App'},
    {'id': 4, 'texto': 'Super bueno el envío express, llegó en 2 horas a Miraflores', 'fuente': 'Google'},
    {'id': 5, 'texto': 'Me cobraron de más y tardaron 1 semana en devolver el dinero', 'fuente': 'Twitter'},
    {'id': 6, 'texto': 'Personal muy amable y precios competitivos, volvería sin duda', 'fuente': 'Google'},
    {'id': 7, 'texto': 'App del supermercado se cae constantemente, muy mala experiencia', 'fuente': 'AppStore'},
    {'id': 8, 'texto': 'Calidad excelente pero precio un poco alto comparado con Metro', 'fuente': 'Google'},
]

POS = {'excelente','super','bueno','rápido','amable','bien','tiempo','sin duda'}
NEG = {'pésima','mala','dañada','tardaron','mal','cae','alto','45 minutos'}

resultados = []
for c in comentarios:
    texto_lower = c['texto'].lower()
    pos = sum(1 for p in POS if p in texto_lower)
    neg = sum(1 for n in NEG if n in texto_lower)
    sentimiento = 'POSITIVO' if pos > neg else ('NEGATIVO' if neg > pos else 'NEUTRO')
    resultados.append({'id': c['id'], 'sentimiento': sentimiento,
                       'palabras': len(c['texto'].split()),
                       'fuente': c['fuente'],
                       'texto': c['texto'][:40]+'...'})

df_sent = pd.DataFrame(resultados)
print('=== ANÁLISIS DE SENTIMIENTO ===')
print(df_sent.to_string(index=False))
print(f'\n📊 Distribución:')
print(df_sent['sentimiento'].value_counts().to_string())

---
## PARTE 4: Arquitectura Big Data para Tu Proyecto (30 min)

### 📝 INSTRUCCIÓN:
Completa el diccionario `mi_arquitectura` con los datos de **TU PROYECTO**.
No uses el ejemplo — usa la empresa y el problema de tu grupo.

In [ ]:
# ── Celda 9: Diseñar arquitectura de TU proyecto ──────────────

mi_arquitectura = {
    # ── INFORMACIÓN DEL PROYECTO ──────────────────────────────
    "proyecto": "Predicción Espacio-Temporal de Congestión Vehicular y Optimización Dinámica de Rutas en Lima Metropolitana",
    "empresa": "Lima Metropolitana — Smart City / Transporte / Gobierno Local",
    "problema": (
        "Lima es la 5ta ciudad más congestionada de América Latina (TomTom 2024). "
        "Los limeños pierden 117 horas/año en tráfico (impacto: S/ 6,200M/año, MTC 2023). "
        "Más de 500K puntos GPS/hora se generan de taxis y buses pero NO se procesan para "
        "optimización en tiempo real: semáforos con tiempos fijos, sin predicción de congestión "
        "anticipada, sin integración de eventos públicos al modelo de tráfico."
    ),

    # ── LAS 5 V's DEL PROYECTO ────────────────────────────────
    "5Vs": {
        "Volumen": "50K vehículos × 960 pts GPS/día = 48M puntos/día → ~3.3 TB/año",
        "Velocidad": "Near Real-time — actualización de rutas cada 2 min; predicción 30 min de anticipación",
        "Variedad": "GPS lat/lon (estructurado), GeoJSON OSM (semi), clima SENAMHI JSON (semi), eventos públicos (no estructurado)",
        "Veracidad": "Señales GPS perdidas, coordenadas fuera de rango, duplicados por re-transmisión",
        "Valor": "Reducir 20% tiempo en viaje = ahorro S/ 1,240M/año; optimizar semáforos y rutas para el MTC",
    },

    # ── ARQUITECTURA TÉCNICA ──────────────────────────────────
    "fuentes_datos": [
        "Simulador GPS 50K vehículos en trayectorias reales de Lima (OSMnx + Faker)",
        "OpenStreetMap Perú — GeoJSON calles e intersecciones de Lima Metropolitana",
        "SENAMHI API — datos climáticos (lluvia, garúa) que afectan velocidad promedio",
        "Agenda de eventos públicos Lima (partidos, conciertos, desfiles — JSON)",
    ],
    "ingesta": "PySpark batch desde Parquet/CSV simulados; OSMnx para descarga grafo OSM; requests para APIs SENAMHI y eventos",
    "almacenamiento": "MongoDB Atlas M0 Free (eventos de tráfico, predicciones); archivos Parquet en Databricks DBFS / Google Drive",
    "procesamiento": "PySpark 3.5 (agregaciones velocidad/densidad por segmento); PySpark GraphX (grafo calles Lima, PageRank cuellos de botella)",
    "analisis": "PySpark MLlib Random Forest temporal (predicción congestión a 30 min, F1 > 0.78); Neo4j Cypher (Dijkstra adaptativo rutas óptimas)",
    "visualizacion": "Folium (mapa interactivo Lima con capas de congestión); Kepler.gl (espacio-temporal); matplotlib para EDA",

    # ── CASOS DE USO ─────────────────────────────────────────
    "casos_uso": [
        "Predicción de congestión: modelo RF predice nivel de tráfico por segmento con 30 min de anticipación (F1 > 0.78)",
        "Ruta óptima adaptativa: Dijkstra en Neo4j recalcula ruta evitando segmentos con alta probabilidad de congestión, cada 2 min",
        "Detección de cuellos de botella: PageRank identifica intersecciones con mayor centralidad de flujo vehicular",
    ],

    # ── CONSIDERACIONES ÉTICAS ───────────────────────────────
    "etica": (
        "El proyecto usa datos GPS SINTÉTICOS (simulados con OSMnx + Faker), no datos reales de vehículos privados, "
        "evitando exposición de privacidad. En producción sería obligatorio: anonimizar IDs de vehículos, "
        "cumplir Ley N.° 29733 de Protección de Datos Personales, y obtener consentimiento "
        "de flotas y conductores para uso analítico de sus trayectorias GPS."
    ),
}

print('=== ARQUITECTURA BIG DATA — SMART TRAFFIC LIMA ===')
print(json.dumps(mi_arquitectura, indent=2, ensure_ascii=False))


In [ ]:
# ── Celda 10: Calcular estimaciones de volumen del PROYECTO ──

# Smart Traffic Lima — 50K vehículos GPS
vehiculos        = 50_000       # vehículos simulados circulando en Lima
registros_x_dia  = 960          # 1 punto GPS/min × 16h operacion = 960 pts/vehiculo/dia
tamanio_registro = 0.2          # KB por punto GPS (lat, lon, timestamp, velocidad, id_vehiculo)

# Calculos
registros_dia  = vehiculos * registros_x_dia
registros_ano  = registros_dia * 365
gb_dia         = (registros_dia * tamanio_registro) / (1024**2)
tb_ano         = gb_dia * 365 / 1024

print('=== ESTIMACIÓN DE VOLUMEN — SMART TRAFFIC LIMA ===')
print(f'Vehículos activos:        {vehiculos:>15,}')
print(f'Puntos GPS/vehículo/día:  {registros_x_dia:>15,}  (1/min x 16h)')
print(f'Registros/día:            {registros_dia:>15,}')
print(f'Registros/año:            {registros_ano:>15,}')
print(f'Tamaño por punto GPS:     {tamanio_registro:>14} KB')
print(f'Tamaño/día:               {gb_dia:>14.1f} GB')
print(f'Tamaño/año:               {tb_ano:>14.1f} TB')
print()
if tb_ano > 1:
    print('ESTO ES BIG DATA — Un solo servidor no puede manejar esto.')
    print('   Necesitamos PySpark distribuido + almacenamiento escalable.')
    print(f'   Los grafos OSM de Lima agregan ~2 GB adicionales de datos geoespaciales.')
else:
    print('Revisar estimacion — el proyecto deberia generar mas volumen.')


---
## ✍️ REFLEXIÓN FINAL (obligatorio — escribe con tus palabras)

Esta celda es evaluada por el docente. Responde honestamente, con tus propias palabras.
No copies de internet — el docente puede detectarlo.

### Mis aprendizajes de esta semana:

**1. ¿Cuál fue el concepto más importante que aprendí?**

El concepto más importante fue entender que Big Data no es solo una cuestión de volumen, sino de cambio de paradigma: cuando el dato crece más allá del techo de una sola máquina, la solución no es comprar más RAM o más disco, sino distribuir el problema entre múltiples nodos. Las 5 V's me dieron un marco concreto para diagnosticar si un problema requiere o no Big Data. En mis 13 años de experiencia trabajando con bases de datos relacionales, he visto repetidamente cómo sistemas bien diseñados para pequeña escala colapsan cuando el volumen se multiplica sin que la arquitectura haya sido diseñada para ello.

**2. ¿Cómo se conecta Big Data con mi trabajo actual?**

Actualmente no estoy empleado, pero en mi anterior posición en la Universidad Norbert Wiener observé de primera mano las consecuencias de no contar con arquitecturas de datos bien establecidas: los sistemas de distintas áreas operaban en silos desconectados, sin gobierno de datos ni trazabilidad. Esto ocasionaba que procesos que podrían manejarse internamente dependieran de terceros o proveedores externos, incrementando costos y exponiendo datos potencialmente sensibles fuera de la organización. Además, varios procesos incumplían parámetros de seguridad porque no había logs centralizados de quién accedía a qué información. Una arquitectura Big Data con Data Lake centralizado, pipelines auditables y control de acceso basado en roles hubiera resuelto exactamente esa problemática.

**3. ¿Qué fue lo más difícil del laboratorio?**

Lo más difícil fue la Parte 4 — el diseño de la arquitectura. No por desconocimiento técnico, sino por la disciplina que requiere justificar cada herramienta en función del problema. La tentación es elegir lo que conoces; el ejercicio obliga a pensar: ¿por qué PySpark y no SQL? ¿Por qué MongoDB y no PostgreSQL? ¿Por qué GraphX para grafos y no joins relacionales? Articular esas respuestas con precisión fue el verdadero desafío, y también lo más valioso.

**4. ¿Por qué elegimos el proyecto que elegimos?**

Elegimos la predicción de tráfico vehicular en Lima porque el problema es urgente, medible y tiene impacto real: Lima es la 5ta ciudad más congestionada de América Latina y cada limeño pierde 117 horas al año en tráfico (casi 5 días completos). El impacto económico es S/ 6,200 millones/año. Los datos existen — miles de taxis y buses generan puntos GPS cada minuto — pero nadie los procesa a escala para optimización en tiempo real. El proyecto nos permite trabajar con grafos (paradigma Big Data poco explorado en cursos convencionales), datos geoespaciales reales de Lima (OpenStreetMap) y machine learning temporal, todo sobre una problemática que cualquier peruano entiende.

**5. ¿Qué pregunta me quedó sin responder?**

¿Cómo maneja Spark Streaming la garantía de exactly-once processing cuando el flujo de puntos GPS incluye duplicados por re-transmisión, y cómo impacta eso en la calidad del grafo de tráfico en tiempo real? ¿Es Delta Lake con transacciones ACID la capa adecuada para materializar el estado del grafo de forma consistente cuando múltiples micro-batches actualizan velocidades por segmento de calle simultáneamente?


In [ ]:
# ── Celda final: Verificación de entrega ──────────────────────
import os

print('=== CHECKLIST DE ENTREGA ===')
checks = [
    ('Nombre escrito en la primera celda',          True),  # Cambia a False si no lo hiciste
    ('Todas las celdas ejecutadas sin error',        True),
    ('Gráfico dashboard_s1.png generado',            os.path.exists('dashboard_s1.png')),
    ('JSON analizado correctamente (Celda 7)',        True),
    ('Análisis de sentimiento completado (Celda 8)', True),
    ('Arquitectura de mi proyecto completada',       True),  # Cambia a False si no lo completaste
    ('Reflexión final escrita con mis palabras',     True),
]

for desc, status in checks:
    icon = '✅' if status else '❌'
    print(f'  {icon}  {desc}')

completados = sum(s for _, s in checks)
print(f'\n{completados}/{len(checks)} items completados')

if completados == len(checks):
    print('\n🎉 ¡Listo para hacer el PR en GitHub!')
    print('   Comando: git add . && git commit -m "[S01] Lab Semana 1 - Tu Nombre"')
else:
    print('\n⚠️  Completa todos los items antes de hacer el PR.')